<a target="_blank" href="https://colab.research.google.com/github/XPOZpublic/xpoz-cookbooks/blob/main/gemini/social-intelligence-with-xpoz-gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Social Intelligence with XPOZ MCP + Gemini

This cookbook shows how to use [XPOZ](https://xpoz.ai) — a social intelligence MCP server with 1.5B+ indexed posts across Twitter, Instagram, Reddit & TikTok — together with Google's Gemini for real-time social media analysis.

XPOZ runs as an MCP (Model Context Protocol) server. You connect to it, call tools like `getTwitterPostsByKeywords` and `countTweets`, and feed the results to Gemini for structured analysis.

**What you'll learn:**
1. Setup: Connect to XPOZ MCP and configure Gemini
2. Brand Sentiment Analysis: Fetch social posts via MCP and classify sentiment with Gemini
3. Competitive Intelligence: Compare two brands' social footprints
4. Influencer Discovery: Find the top voices on any topic
5. Real-time Event Monitoring: Social overlay for prediction markets

**Prerequisites:**
- XPOZ API key (free at [xpoz.ai](https://xpoz.ai) — 100K results/month)
- Google AI API key ([aistudio.google.com](https://aistudio.google.com))

## 1. Setup

In [ ]:
# Install dependencies
!pip install mcp google-generativeai

In [ ]:
import os

# Set your API keys (or use environment variables)
os.environ["XPOZ_API_KEY"] = "your-xpoz-api-key"    # Get at xpoz.ai
os.environ["GOOGLE_API_KEY"] = "your-google-api-key"  # Get at aistudio.google.com

In [ ]:
import google.generativeai as genai
import json
from datetime import datetime, timedelta
from mcp.client.streamable_http import streamablehttp_client
from mcp import ClientSession

# Connect to XPOZ MCP server
transport = await streamablehttp_client(
    url="https://mcp.xpoz.ai/mcp",
    headers={"Authorization": f"Bearer {os.environ['XPOZ_API_KEY']}"}
).__aenter__()
read, write, _ = transport

session = ClientSession(read, write)
await session.__aenter__()
await session.initialize()

# Initialize Gemini
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
gemini = genai.GenerativeModel("gemini-2.0-flash")

# Helper: get date N days ago
def days_ago(n: int) -> str:
    return (datetime.now() - timedelta(days=n)).strftime("%Y-%m-%d")

# Helper: call XPOZ MCP tool and parse response
async def call_xpoz(tool_name: str, params: dict) -> dict:
    """Call an XPOZ MCP tool and return parsed response."""
    result = await session.call_tool(tool_name, params)
    return json.loads(result.content[0].text)

# Verify connection
tools = await session.list_tools()
print(f"Connected to XPOZ MCP — {len(tools.tools)} tools available")
print(f"Sample tools: {', '.join(t.name for t in tools.tools[:6])}")

## 2. Brand Sentiment Analysis

Fetch recent social media posts about a brand via XPOZ MCP and use Gemini to classify sentiment, extract themes, and identify risks.

In [ ]:
# Step 1: Fetch social data via XPOZ MCP
BRAND = "NVIDIA"

# Count total volume
count_result = await call_xpoz("countTweets", {
    "phrase": BRAND,
    "startDate": days_ago(7)
})
tweet_count = count_result["data"]["count"]

# Search Twitter posts
twitter_result = await call_xpoz("getTwitterPostsByKeywords", {
    "query": BRAND,
    "startDate": days_ago(7),
    "limit": 50,
    "fields": ["text", "authorUsername", "likeCount", "retweetCount", "createdAtDate"]
})
tweets = twitter_result["data"]["results"]

# Search Reddit posts
reddit_result = await call_xpoz("getRedditPostsByKeywords", {
    "query": BRAND,
    "startDate": days_ago(7),
    "limit": 30,
    "fields": ["title", "selftext", "authorUsername", "score", "subredditName", "createdAtDate"]
})
reddit_posts = reddit_result["data"]["results"]

print(f"Found {len(tweets)} tweets, {len(reddit_posts)} Reddit posts")
print(f"Total tweet volume (7 days): {tweet_count:,}")

In [ ]:
# Step 2: Analyze with Gemini
tweets_text = "\n".join([
    f"@{p['authorUsername']}: {p['text']} [likes:{p.get('likeCount',0)}, RTs:{p.get('retweetCount',0)}]"
    for p in tweets
])

reddit_text = "\n".join([
    f"r/{p.get('subredditName','')} — {p['title']}: {(p.get('selftext') or '')[:200]} [score:{p.get('score',0)}]"
    for p in reddit_posts
])

response = gemini.generate_content(
    f"""Analyze the social media sentiment for {BRAND} based on these posts from the last 7 days.
Total tweet volume: {tweet_count:,}

Twitter posts:
{tweets_text}

Reddit posts:
{reddit_text}

Provide:
1. Overall Sentiment (Bullish/Bearish/Neutral with confidence %)
2. Key Themes (top 5 with sentiment for each)
3. Risk Signals (emerging issues or negative trends)
4. Opportunities (positive trends to capitalize on)
5. Most Impactful Posts (3 highest-engagement posts)"""
)

print(response.text)

## 3. Competitive Intelligence

Compare two brands' social media presence — volume, sentiment, themes, and share of voice.

In [ ]:
# Compare two brands using XPOZ MCP
BRAND_A = "NVIDIA"
BRAND_B = "AMD"

async def fetch_brand_data(brand: str):
    tweets_r = await call_xpoz("getTwitterPostsByKeywords", {
        "query": brand, "startDate": days_ago(7), "limit": 50,
        "fields": ["text", "authorUsername", "likeCount", "retweetCount"]
    })
    reddit_r = await call_xpoz("getRedditPostsByKeywords", {
        "query": brand, "startDate": days_ago(7), "limit": 30,
        "fields": ["title", "selftext", "score", "subredditName"]
    })
    count_r = await call_xpoz("countTweets", {
        "phrase": brand, "startDate": days_ago(7)
    })
    tweets = tweets_r["data"]["results"]
    return {
        "brand": brand,
        "tweet_count": count_r["data"]["count"],
        "tweets": [p["text"] for p in tweets],
        "reddit_posts": [f"{p['title']}: {(p.get('selftext') or '')[:100]}" for p in reddit_r["data"]["results"]],
        "avg_engagement": sum(p.get("likeCount", 0) or 0 for p in tweets) / max(len(tweets), 1)
    }

data_a = await fetch_brand_data(BRAND_A)
data_b = await fetch_brand_data(BRAND_B)

total_vol = data_a['tweet_count'] + data_b['tweet_count']
print(f"{BRAND_A}: {data_a['tweet_count']:,} tweets, avg {data_a['avg_engagement']:.0f} likes")
print(f"{BRAND_B}: {data_b['tweet_count']:,} tweets, avg {data_b['avg_engagement']:.0f} likes")
print(f"Share of Voice: {BRAND_A} {data_a['tweet_count']/max(total_vol,1)*100:.0f}% / {BRAND_B} {data_b['tweet_count']/max(total_vol,1)*100:.0f}%")

In [ ]:
# Competitive analysis with Gemini
response = gemini.generate_content(
    f"""Compare these two brands' social media presence over the last 7 days:

{BRAND_A}:
- Tweet volume: {data_a['tweet_count']:,}
- Sample tweets: {json.dumps(data_a['tweets'][:10])}
- Reddit: {json.dumps(data_a['reddit_posts'][:5])}

{BRAND_B}:
- Tweet volume: {data_b['tweet_count']:,}
- Sample tweets: {json.dumps(data_b['tweets'][:10])}
- Reddit: {json.dumps(data_b['reddit_posts'][:5])}

Provide a competitive intelligence brief:
1. Share of Voice comparison
2. Sentiment comparison
3. Key differentiators in public perception
4. Emerging narratives that could shift the landscape
5. Strategic recommendations for each brand"""
)

print(response.text)

## 4. Influencer Discovery

Find the most influential voices discussing a topic across Twitter using XPOZ MCP.

In [ ]:
TOPIC = '"artificial intelligence" AND ethics'

# Find users who post about this topic via MCP
users_result = await call_xpoz("getTwitterUsersByKeywords", {
    "query": TOPIC,
    "startDate": days_ago(30),
    "limit": 20,
    "fields": ["username", "name", "followersCount", "description", "verified",
               "relevantTweetsCount", "relevantTweetsImpressionsSum"]
})
users = users_result["data"]["results"]

print(f"Found {len(users)} influencers on '{TOPIC}':\n")
for u in sorted(users, key=lambda x: x.get("followersCount", 0) or 0, reverse=True)[:10]:
    verified = " \u2713" if u.get("verified") else ""
    print(f"@{u['username']}{verified} — {(u.get('followersCount') or 0):,} followers")
    print(f"  {(u.get('description') or '')[:80]}")
    print(f"  {u.get('relevantTweetsCount', 0) or 0} relevant posts, {(u.get('relevantTweetsImpressionsSum') or 0):,} impressions")
    print()

## 5. Real-time Event Monitoring

Monitor social sentiment around real-world events — useful for prediction markets, trading signals, and crisis detection.

In [ ]:
EVENT = "Bitcoin price"
QUERY_YES = '"bitcoin" AND ("bullish" OR "moon" OR "buy" OR "breakout" OR "rally" OR "ATH")'
QUERY_NO = '"bitcoin" AND ("bearish" OR "crash" OR "sell" OR "dump" OR "bubble" OR "overvalued")'

# Count bullish vs bearish via MCP
bullish_r = await call_xpoz("countTweets", {"phrase": QUERY_YES, "startDate": days_ago(3)})
bearish_r = await call_xpoz("countTweets", {"phrase": QUERY_NO, "startDate": days_ago(3)})

bullish_count = bullish_r["data"]["count"]
bearish_count = bearish_r["data"]["count"]
total = bullish_count + bearish_count

print(f"Social Sentiment for '{EVENT}' (last 3 days):")
print(f"  Bullish: {bullish_count:,} posts ({bullish_count/max(total,1)*100:.0f}%)")
print(f"  Bearish: {bearish_count:,} posts ({bearish_count/max(total,1)*100:.0f}%)")
print(f"  Sentiment Ratio: {bullish_count/max(bearish_count,1):.1f}x bullish")

In [ ]:
# Get sample posts and analyze with Gemini
bullish_r = await call_xpoz("getTwitterPostsByKeywords", {
    "query": QUERY_YES, "startDate": days_ago(3), "limit": 20,
    "fields": ["text", "authorUsername", "likeCount", "retweetCount"]
})
bearish_r = await call_xpoz("getTwitterPostsByKeywords", {
    "query": QUERY_NO, "startDate": days_ago(3), "limit": 20,
    "fields": ["text", "authorUsername", "likeCount", "retweetCount"]
})

bullish_posts = bullish_r["data"]["results"]
bearish_posts = bearish_r["data"]["results"]

response = gemini.generate_content(
    f"""Analyze the social sentiment around "{EVENT}" to produce a prediction signal.

Volume (3 days): {bullish_count:,} bullish vs {bearish_count:,} bearish posts.

Bullish posts (sample):
{chr(10).join(f'@{p["authorUsername"]}: {p["text"]} [likes:{p.get("likeCount",0)}]' for p in bullish_posts[:10])}

Bearish posts (sample):
{chr(10).join(f'@{p["authorUsername"]}: {p["text"]} [likes:{p.get("likeCount",0)}]' for p in bearish_posts[:10])}

Provide:
1. Sentiment Signal: BULLISH / BEARISH / NEUTRAL (with confidence 1-10)
2. Key Arguments from each side
3. Influential Voices driving each narrative
4. Catalysts: Recent events driving sentiment
5. Risk Assessment: What could flip the sentiment?"""
)

print(response.text)

## 6. Generate HTML Report

Save the brand sentiment analysis as a styled HTML report you can open in a browser.

In [ ]:
import re

# Use the brand sentiment analysis from Section 2
analysis_text = response.text

# Format tweets table
tweets_html = ""
for t in sorted(tweets, key=lambda x: (x.get('likeCount', 0) or 0), reverse=True)[:15]:
    tweets_html += f"""
    <tr>
      <td style="font-size:.85em">@{t['authorUsername']}</td>
      <td style="font-size:.85em">{t['text'][:120]}</td>
      <td style="text-align:right">{t.get('likeCount',0)}</td>
      <td style="text-align:right">{t.get('retweetCount',0)}</td>
    </tr>"""

# Format analysis for HTML
analysis_escaped = analysis_text.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
analysis_escaped = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', analysis_escaped)
analysis_escaped = analysis_escaped.replace('\n\n', '<br><br>').replace('\n', '<br>')

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>{BRAND} Brand Sentiment — Gemini Report</title>
<style>
  *{{margin:0;padding:0;box-sizing:border-box}}
  body{{background:#0f172a;color:#e2e8f0;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;padding:32px;max-width:1100px;margin:0 auto}}
  h1{{font-size:2em;margin-bottom:4px;background:linear-gradient(135deg,#a855f7,#ec4899);-webkit-background-clip:text;-webkit-text-fill-color:transparent}}
  h2{{color:#a855f7;margin:28px 0 12px;font-size:1.2em;border-bottom:1px solid #1e293b;padding-bottom:8px}}
  .header{{text-align:center;margin-bottom:36px}}
  .meta{{color:#64748b;font-size:.9em;margin-top:8px}}
  .stat-grid{{display:grid;grid-template-columns:repeat(3,1fr);gap:16px;margin:20px 0}}
  .stat{{background:#1e293b;border-radius:12px;padding:20px;text-align:center}}
  .stat .value{{font-size:2em;font-weight:800;color:#a855f7}}
  .stat .label{{color:#64748b;font-size:.85em;margin-top:4px}}
  .analysis{{background:#1e293b;border-radius:12px;padding:24px;margin:16px 0;line-height:1.7;color:#cbd5e1}}
  .analysis strong{{color:#e2e8f0}}
  table{{width:100%;border-collapse:collapse;margin:12px 0}}
  th,td{{padding:8px 12px;text-align:left;border-bottom:1px solid #1e293b}}
  th{{color:#94a3b8;font-size:.8em;text-transform:uppercase}}
  .footer{{text-align:center;color:#475569;font-size:.8em;margin-top:40px;padding-top:20px;border-top:1px solid #1e293b}}
  .badge{{display:inline-block;background:#a855f720;color:#a855f7;padding:4px 12px;border-radius:20px;font-size:.85em;font-weight:600}}
</style>
</head>
<body>
<div class="header">
  <h1>{BRAND} — Brand Sentiment Report</h1>
  <div style="margin:8px 0"><span class="badge">XPOZ MCP</span> <span class="badge">Gemini</span></div>
  <div class="meta">Data collected via XPOZ MCP (countTweets, getTwitterPostsByKeywords, getRedditPostsByKeywords) · {datetime.now().strftime('%Y-%m-%d %H:%M')}</div>
</div>
<div class="stat-grid">
  <div class="stat"><div class="value">{tweet_count:,}</div><div class="label">Total Tweets (7d)</div></div>
  <div class="stat"><div class="value">{len(tweets) + len(reddit_posts)}</div><div class="label">Posts Analyzed</div></div>
  <div class="stat"><div class="value">{sum((t.get('likeCount',0) or 0) for t in tweets):,}</div><div class="label">Total Engagement</div></div>
</div>
<h2>Gemini Analysis</h2>
<div class="analysis">{analysis_escaped}</div>
<h2>Sample Posts (by engagement)</h2>
<table>
  <thead><tr><th>Author</th><th>Text</th><th style="text-align:right">Likes</th><th style="text-align:right">RTs</th></tr></thead>
  <tbody>{tweets_html}</tbody>
</table>
<div class="footer">XPOZ Social Intelligence MCP · Gemini · <a href="https://xpoz.ai/install" style="color:#a855f7">xpoz.ai/install</a></div>
</body></html>"""

report_path = f"{BRAND.lower()}-sentiment-gemini.html"
with open(report_path, "w") as f:
    f.write(html)
print(f"HTML report saved: {report_path}")
print(f"Open with: open {report_path}")

In [ ]:
# Clean up MCP session
await session.__aexit__(None, None, None)
print("Done! MCP session closed.")